In [4]:
import pandas as pd
import numpy as np

df_clean = pd.read_csv('../data/processed/application_train_clean.csv')
df_clean.shape

(307511, 79)

In [5]:
def create_anomaly_features(df):
    df = df.copy()
    
    # 1. Rasio income terhadap credit yang ekstrim (udah ada CREDIT_INCOME_RATIO)
    # tambahin flag ekstrimnya
    df['FLAG_HIGH_CREDIT_INCOME_RATIO'] = (df['CREDIT_INCOME_RATIO'] > df['CREDIT_INCOME_RATIO'].quantile(0.99)).astype(int)
    
    # 2. Umur vs lama kerja ga masuk akal
    # DAYS_BIRTH negatif (hari sejak lahir), DAYS_EMPLOYED negatif (hari sejak kerja)
    df['AGE_YEARS'] = -df['DAYS_BIRTH'] / 365
    df['EMPLOYED_YEARS'] = -df['DAYS_EMPLOYED'] / 365
    df['FLAG_EMPLOYED_LONGER_THAN_POSSIBLE'] = (
        df['EMPLOYED_YEARS'] > (df['AGE_YEARS'] - 14)  # asumsi mulai kerja minimal umur 14
    ).astype(int)
    
    # 3. Income ekstrim outlier
    df['FLAG_INCOME_OUTLIER'] = (df['AMT_INCOME_TOTAL'] > df['AMT_INCOME_TOTAL'].quantile(0.999)).astype(int)
    
    # 4. Jumlah dokumen yang inconsistent (contoh: submit banyak dokumen tapi income ga match)
    doc_cols = [c for c in df.columns if c.startswith('FLAG_DOCUMENT_')]
    df['TOTAL_DOCUMENTS_SUBMITTED'] = df[doc_cols].sum(axis=1)
    
    # 5. Terlalu banyak inquiry ke credit bureau dalam waktu singkat (credit hunger)
    df['FLAG_HIGH_BUREAU_INQUIRY'] = (df['AMT_REQ_CREDIT_BUREAU_YEAR'] > df['AMT_REQ_CREDIT_BUREAU_YEAR'].quantile(0.99)).astype(int)
    
    return df

In [6]:
df_anomaly = create_anomaly_features(df_clean)

flag_cols = [c for c in df_anomaly.columns if c.startswith('FLAG_') and c not in df_clean.columns]
for col in flag_cols:
    print(col, df_anomaly[col].sum(), f"({df_anomaly[col].mean()*100:.2f}%)")

FLAG_HIGH_CREDIT_INCOME_RATIO 3076 (1.00%)
FLAG_EMPLOYED_LONGER_THAN_POSSIBLE 0 (0.00%)
FLAG_INCOME_OUTLIER 278 (0.09%)
FLAG_HIGH_BUREAU_INQUIRY 1237 (0.40%)


In [7]:
df_anomaly[['AGE_YEARS', 'EMPLOYED_YEARS']].describe()

,AGE_YEARS,EMPLOYED_YEARS
count,307511.000000,307511.000000
mean,43.936973,6.168784
std,11.956133,5.852585
min,20.517808,-0.000000
25%,34.008219,2.556164
50%,43.150685,4.515068
75%,53.923288,7.561644
max,69.120548,49.073973


In [8]:
!pip install scikit-learn --quiet

In [9]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Pilih fitur numerik utama buat deteksi anomali
# gabungan fitur asli + fitur red flag yang udah kita bikin
iso_features = [
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO',
    'AGE_YEARS', 'EMPLOYED_YEARS',
    'AMT_REQ_CREDIT_BUREAU_YEAR', 'TOTAL_DOCUMENTS_SUBMITTED'
]

X_iso = df_anomaly[iso_features].copy()

# Scaling penting buat Isolation Forest karena beda skala (income jutaan vs dokumen 0-20)
scaler = StandardScaler()
X_iso_scaled = scaler.fit_transform(X_iso)

# Train Isolation Forest
# contamination = perkiraan proporsi anomali, kita mulai dengan asumsi 2%
iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.02,
    random_state=42,
    n_jobs=-1
)

df_anomaly['ANOMALY_PRED'] = iso_forest.fit_predict(X_iso_scaled)
df_anomaly['ANOMALY_SCORE'] = iso_forest.decision_function(X_iso_scaled)

# -1 = anomali, 1 = normal (output asli sklearn), kita convert ke lebih intuitif
df_anomaly['IS_ANOMALY'] = (df_anomaly['ANOMALY_PRED'] == -1).astype(int)

print(df_anomaly['IS_ANOMALY'].value_counts())
print(df_anomaly['IS_ANOMALY'].mean())

IS_ANOMALY
0    301360
1      6151
Name: count, dtype: int64
0.02000253649462946


In [10]:
print(df_anomaly.groupby('IS_ANOMALY')['TARGET'].mean())

IS_ANOMALY
0    0.081471
1    0.044383
Name: TARGET, dtype: float64


In [11]:
# Lihat statistik fitur di grup anomali vs normal
for col in iso_features:
    print(col)
    print(df_anomaly.groupby('IS_ANOMALY')[col].mean())
    print('---')

AMT_INCOME_TOTAL
IS_ANOMALY
0    165882.732766
1    311623.575751
Name: AMT_INCOME_TOTAL, dtype: float64
---
AMT_CREDIT
IS_ANOMALY
0    5.776059e+05
1    1.648476e+06
Name: AMT_CREDIT, dtype: float64
---
AMT_ANNUITY
IS_ANOMALY
0    26466.538424
1    58559.939034
Name: AMT_ANNUITY, dtype: float64
---
AMT_GOODS_PRICE
IS_ANOMALY
0    5.179965e+05
1    1.533859e+06
Name: AMT_GOODS_PRICE, dtype: float64
---
CREDIT_INCOME_RATIO
IS_ANOMALY
0    3.840484
1    9.694046
Name: CREDIT_INCOME_RATIO, dtype: float64
---
ANNUITY_INCOME_RATIO
IS_ANOMALY
0    0.177854
1    0.331565
Name: ANNUITY_INCOME_RATIO, dtype: float64
---
AGE_YEARS
IS_ANOMALY
0    43.881604
1    46.649689
Name: AGE_YEARS, dtype: float64
---
EMPLOYED_YEARS
IS_ANOMALY
0    6.106552
1    9.217769
Name: EMPLOYED_YEARS, dtype: float64
---
AMT_REQ_CREDIT_BUREAU_YEAR
IS_ANOMALY
0    1.645597
1    1.538124
Name: AMT_REQ_CREDIT_BUREAU_YEAR, dtype: float64
---
TOTAL_DOCUMENTS_SUBMITTED
IS_ANOMALY
0    0.927356
1    1.067306
Name: TOTAL_DOCU

In [12]:
import sys
sys.path.append('../src')
from anomaly_detection import create_anomaly_features, fit_anomaly_detector, apply_anomaly_detector

df_anomaly = create_anomaly_features(df_clean)
iso_forest, scaler = fit_anomaly_detector(df_anomaly, contamination=0.02)
df_final = apply_anomaly_detector(df_anomaly, iso_forest, scaler)

df_final['IS_ANOMALY'].value_counts()

IS_ANOMALY
0    301360
1      6151
Name: count, dtype: int64

In [13]:
import joblib
import os

os.makedirs('../models', exist_ok=True)
joblib.dump(iso_forest, '../models/isolation_forest.pkl')
joblib.dump(scaler, '../models/anomaly_scaler.pkl')

df_final.to_csv('../data/processed/application_train_with_anomaly.csv', index=False)